# Week 4 - Day 5: LangGraph Orchestration & Tool Calling

This notebook turns Days 1-4 into an actual LangGraph agent: a typed state schema, a
compiled state machine that routes between conversation stages, six tools wrapping the
mock Calendar/Email/CRM/property layers built on Days 2 and 4, guardrails that block
unsafe recommendations and bookings, and an annotated execution trace for every call.

`pip install langgraph langchain-core` is the only new dependency versus Days 1-4.

# Task 1 - LangGraph State Design

`VoiceAgentState` is the single object every node reads and writes. Two fields use
`Annotated[..., operator.add]` so LangGraph *appends* to them on every node return instead
of overwriting â€” `conversation_history` and `execution_trace` need the full history, not
just the latest node's output; everything else (budget, intent, appointment_status, ...)
is a plain overwrite field since only the latest value matters.

`last_action_result` exists for a subtle reason: `appointment_id` stays set for the rest of
the call once a visit is booked, so a *later* rescheduling attempt that gets refused would
otherwise look indistinguishable from a successful one if the router only checked
`appointment_id`. `last_action_result` is reset every turn by the node that just ran, so the
router can tell "this turn succeeded" from "an appointment exists from an earlier turn."

`utterance_queue` is the one field that doesn't belong in a production deployment as-is â€”
it stands in for the live STT stream so this notebook can drive full, deterministic,
multi-turn conversations without a live phone call or an LLM API key, the same convention
Day 3's local adapters used.

In [1]:
from typing import TypedDict, Optional, Literal
import operator
from typing_extensions import Annotated


class Message(TypedDict):
    role: str  # "caller" | "agent"
    content: str


class ToolCallRecord(TypedDict):
    tool: str
    input: dict
    output: dict


class TraceEntry(TypedDict):
    node: str
    entered_intent: Optional[str]
    appointment_status: str
    note: str


class VoiceAgentState(TypedDict):
    # Conversation history: every reducer below appends rather than overwrites, so each
    # graph invocation can add to the running call record instead of replacing it.
    conversation_history: Annotated[list[Message], operator.add]

    # Caller-provided facts, accumulated as they're mentioned across turns.
    user_profile: dict            # client_name, client_phone, employee_id (once assigned)
    property_preferences: dict    # city, area, property_type, bedrooms, purpose
    budget: Optional[int]

    # Routing state
    intent: Optional[str]
    pending_clarification: Optional[str]

    # What the graph has learned/decided so far
    matched_property_id: Optional[str]
    appointment_id: Optional[str]
    selected_slot: Optional[dict]
    appointment_status: Literal["none", "pending", "booked", "rescheduled", "cancelled"]

    # Per-turn outcome of the action node just visited (booking/rescheduling/cancellation),
    # so the router can tell a *successful* action apart from a *refused* one without
    # confusing it with an appointment_id left over from an earlier turn.
    last_action_result: Optional[Literal["success", "failed"]]

    # Every tool call the graph makes, for audit/debugging (separate from execution_trace,
    # which logs node transitions rather than tool I/O).
    tool_outputs: Annotated[list[ToolCallRecord], operator.add]

    # Task 5: one entry per node the graph visits, in order.
    execution_trace: Annotated[list[TraceEntry], operator.add]

    # Driver-only field: the remaining simulated caller turns for this call.
    # A real deployment replaces this with the live STT stream; the graph logic
    # downstream of intent_detection doesn't know or care where the utterance came from.
    utterance_queue: list[str]


def initial_state(client_name: str, client_phone: str, utterances: list[str]) -> VoiceAgentState:
    return VoiceAgentState(
        conversation_history=[],
        user_profile={"client_name": client_name, "client_phone": client_phone},
        property_preferences={},
        budget=None,
        intent=None,
        pending_clarification=None,
        matched_property_id=None,
        appointment_id=None,
        selected_slot=None,
        appointment_status="none",
        last_action_result=None,
        tool_outputs=[],
        execution_trace=[],
        utterance_queue=list(utterances),
    )


sample_state = initial_state("Bilal Farooq", "+92-321-5551042", ["Budget 3 crore hai, DHA mein ghar chahiye"])
print("State fields:", list(sample_state.keys()))
print("Initial appointment_status:", sample_state["appointment_status"])
print("Utterance queue:", sample_state["utterance_queue"])

State fields: ['conversation_history', 'user_profile', 'property_preferences', 'budget', 'intent', 'pending_clarification', 'matched_property_id', 'appointment_id', 'selected_slot', 'appointment_status', 'last_action_result', 'tool_outputs', 'execution_trace', 'utterance_queue']
Initial appointment_status: none
Utterance queue: ['Budget 3 crore hai, DHA mein ghar chahiye']


# Task 2 - Graph Design

Nine nodes, matching the brief exactly: `greeting`, `intent_detection`, `rag`,
`recommendation`, `booking`, `rescheduling`, `cancellation`, `email`, `goodbye`.

Routing rules:
- `greeting` always flows into `intent_detection`.
- `intent_detection` branches on the parsed intent via `route_after_intent` â€” including a
  self-loop back to itself for an `unclear` intent, so the agent can read the caller's next
  turn (in production: ask a clarifying question first) instead of guessing a path forward.
- `rag` and `recommendation` are informational â€” they always return to `intent_detection` so
  the call can continue after an answer or a suggestion.
- `booking`, `rescheduling`, and `cancellation` only proceed to `email` when
  `last_action_result == "success"` (`route_after_action`); a refused action loops straight
  back to `intent_detection` instead, since Task 4's guardrails mean these nodes don't always
  produce an appointment.
- `email` always returns to `intent_detection`; `goodbye` is the only edge to `END`.

The node functions below call tool functions (`search_property_tool`,
`availability_checker_tool`, etc.) that aren't defined until Task 3. That's safe here: Python
only resolves those names when a node actually *runs*, and `graph_builder.compile()` doesn't
run any node â€” it just wires the structure together. The mermaid diagram printed at the end
is generated directly from the compiled graph object, not drawn by hand, so it's guaranteed
to match the code.

In [2]:
from langgraph.graph import StateGraph, END
import re


def log(state: VoiceAgentState, node: str, note: str) -> dict:
    """Shared helper every node uses to append its Task-5 trace entry."""
    return {"execution_trace": [{
        "node": node, "entered_intent": state.get("intent"),
        "appointment_status": state["appointment_status"], "note": note,
    }]}


CITY_KEYWORDS = {"dha": "DHA Phase 6", "gulberg": "Gulberg III", "bahria": "Bahria Town"}
CRORE_RE = re.compile(r"(\d+(?:\.\d+)?)\s*crore", re.IGNORECASE)
TIME_RE = re.compile(r"(\d{1,2})\s*(am|pm)", re.IGNORECASE)


def greeting_node(state: VoiceAgentState) -> dict:
    greeting = "Assalam-o-alaikum, main property team ka AI assistant hoon. Aap kis type ki property dekh rahe hain?"
    return {
        "conversation_history": [{"role": "agent", "content": greeting}],
        **log(state, "greeting", "Opened the call."),
    }


def intent_detection_node(state: VoiceAgentState) -> dict:
    queue = state["utterance_queue"]
    if not queue:
        return {"intent": "goodbye", **log(state, "intent_detection", "No more caller turns; ending call.")}

    utterance, remaining = queue[0], queue[1:]
    text = utterance.lower()
    updates: dict = {"utterance_queue": remaining, "conversation_history": [{"role": "caller", "content": utterance}]}

    crore_match = CRORE_RE.search(text)
    if crore_match:
        updates["budget"] = int(float(crore_match.group(1)) * 10_000_000)
    prefs = dict(state["property_preferences"])
    for keyword, area in CITY_KEYWORDS.items():
        if keyword in text:
            prefs["area"] = area
    bed_match = re.search(r"(\d+)\s*bedroom", text)
    if bed_match:
        prefs["bedrooms"] = int(bed_match.group(1))
    if prefs != state["property_preferences"]:
        updates["property_preferences"] = prefs

    if "cheap" in text or "sasti" in text:
        intent = "recommendation_request"  # "us se sasti koi option?" -> refine and re-recommend
    elif any(w in text for w in ("cancel", "kardo")) and "cancel" in text:
        intent = "cancel"
    elif any(w in text for w in ("reschedule", "different time", "move the visit")):
        intent = "reschedule"
    elif any(w in text for w in ("book", "confirm kar", "schedule the visit", "visit book")):
        intent = "book"
    elif any(w in text for w in ("plot size", "possession", "amenit", "documents required", "school",
                                  "guarantee", "return", "invest", "price change")):
        intent = "property_question"
    elif any(w in text for w in ("option", "ghar chahiye", "koi property", "recommend", "dikhao", "available")):
        intent = "recommendation_request"
    elif any(w in text for w in ("bye", "thank you", "shukriya", "that's all")):
        intent = "goodbye"
    else:
        intent = "unclear"

    # A caller answering "which alternative slot works" rarely repeats the word "book" â€”
    # treat that reply as a continuation of the booking flow it's answering, not a new topic.
    if intent == "unclear" and state.get("pending_clarification") == "awaiting_alternative_slot_choice" and TIME_RE.search(text):
        intent = "book"

    updates["intent"] = intent
    return {**updates, **log(state, "intent_detection", f"Parsed intent={intent} from: \"{utterance}\"")}


def rag_node(state: VoiceAgentState) -> dict:
    utterance = state["conversation_history"][-1]["content"]
    result = rag_search_tool.invoke({"query": utterance})
    if result["evidence_id"] is None:
        # Task 4 validation: no grounded evidence -> ask, don't guess.
        reply = "Sorry, is baare mein mujhe verified information nahi mil rahi. Main specialist se confirm karwa kar callback arrange kar sakta hoon â€” theek hai?"
        updates = {"pending_clarification": "awaiting_specialist_callback_confirmation"}
    else:
        reply = result["answer"]
        updates = {"pending_clarification": None}
    return {
        "conversation_history": [{"role": "agent", "content": reply}],
        "tool_outputs": [{"tool": "rag_search", "input": {"query": utterance}, "output": result}],
        **updates,
        **log(state, "rag", f"RAG lookup grounded={result['evidence_id'] is not None}"),
    }


def recommendation_node(state: VoiceAgentState) -> dict:
    search_result = search_property_tool.invoke({
        "area": state["property_preferences"].get("area"),
        "bedrooms": state["property_preferences"].get("bedrooms"),
        "max_budget": state["budget"],
    })
    available = availability_checker_tool.invoke({"property_ids": [p["property_id"] for p in search_result["matches"]]})
    available_ids = {pid for pid, is_available in available["availability"].items() if is_available}
    # Task 4 validation: never recommend a property that failed the live availability check,
    # even if it matched on price/area/bedrooms.
    viable = [p for p in search_result["matches"] if p["property_id"] in available_ids]

    tool_log = [
        {"tool": "search_property", "input": {"preferences": state["property_preferences"], "budget": state["budget"]}, "output": search_result},
        {"tool": "availability_checker", "input": {"property_ids": [p["property_id"] for p in search_result["matches"]]}, "output": available},
    ]

    if not viable:
        reply = ("Is budget aur area mein abhi koi available option nahi hai. Kya aap budget thora barha saktay hain, "
                 "ya nearby area consider karain gay?")
        return {
            "conversation_history": [{"role": "agent", "content": reply}],
            "tool_outputs": tool_log, "matched_property_id": None,
            "pending_clarification": "awaiting_budget_or_area_adjustment",
            **log(state, "recommendation", "No available match within stated constraints; asked for clarification instead of guessing."),
        }

    best = viable[0]
    reply = (f"{best['name']} available hai {best['area']} mein, {best['bedrooms']} bedroom, "
             f"asking price {best['price']:,} PKR. Visit book karwana chahain gay?")
    return {
        "conversation_history": [{"role": "agent", "content": reply}],
        "tool_outputs": tool_log, "matched_property_id": best["property_id"], "pending_clarification": None,
        **log(state, "recommendation", f"Recommended {best['property_id']} (available, within budget)."),
    }


def booking_node(state: VoiceAgentState) -> dict:
    if not state["matched_property_id"]:
        reply = "Booking se pehle mujhe property confirm karni hogi â€” pehle main aap ko options dikhata hoon."
        return {"conversation_history": [{"role": "agent", "content": reply}], "last_action_result": "failed",
                **log(state, "booking", "Refused: no property matched yet.")}

    utterance = state["conversation_history"][-1]["content"]
    time_match = TIME_RE.search(utterance.lower())
    requested_slot = time_match.group(0) if time_match else "next available slot"

    slot_check = availability_checker_tool.invoke({"property_id": state["matched_property_id"], "slot": requested_slot})
    if not slot_check["slot_available"]:
        reply = (f"{requested_slot} par slot already book hai. Available slots hain: "
                 f"{', '.join(slot_check['alternative_slots'])}. In mein se konsa theek rahega?")
        return {
            "conversation_history": [{"role": "agent", "content": reply}],
            "tool_outputs": [{"tool": "availability_checker", "input": {"slot": requested_slot}, "output": slot_check}],
            "last_action_result": "failed", "pending_clarification": "awaiting_alternative_slot_choice",
            **log(state, "booking", "Refused: requested slot unavailable (offered alternatives instead of guessing)."),
        }

    calendar_result = calendar_tool.invoke({
        "operation": "create", "client_name": state["user_profile"]["client_name"],
        "client_phone": state["user_profile"]["client_phone"], "property_id": state["matched_property_id"],
        "slot": requested_slot,
    })
    crm_tool.invoke({"operation": "log_appointment", "appointment_id": calendar_result["appointment_id"],
                      "action": "booked", "detail": f"Slot {requested_slot}"})
    reply = f"Confirm ho gaya â€” {requested_slot} par aap ki visit book hai. Confirmation email employee ko chali jayegi."
    return {
        "conversation_history": [{"role": "agent", "content": reply}],
        "tool_outputs": [
            {"tool": "availability_checker", "input": {"slot": requested_slot}, "output": slot_check},
            {"tool": "calendar", "input": {"operation": "create"}, "output": calendar_result},
        ],
        "appointment_id": calendar_result["appointment_id"], "selected_slot": {"slot": requested_slot},
        "appointment_status": "booked", "last_action_result": "success", "pending_clarification": None,
        **log(state, "booking", f"Booked {calendar_result['appointment_id']} at {requested_slot}."),
    }


def rescheduling_node(state: VoiceAgentState) -> dict:
    if not state["appointment_id"]:
        reply = "Aap ki koi active appointment nahi mili is number par."
        return {"conversation_history": [{"role": "agent", "content": reply}], "last_action_result": "failed",
                **log(state, "rescheduling", "Refused: no existing appointment to reschedule.")}

    utterance = state["conversation_history"][-1]["content"]
    time_match = TIME_RE.search(utterance.lower())
    new_slot = time_match.group(0) if time_match else "next available slot"
    slot_check = availability_checker_tool.invoke({"property_id": state["matched_property_id"], "slot": new_slot})
    if not slot_check["slot_available"]:
        reply = f"{new_slot} bhi available nahi hai. Try: {', '.join(slot_check['alternative_slots'])}."
        return {
            "conversation_history": [{"role": "agent", "content": reply}],
            "tool_outputs": [{"tool": "availability_checker", "input": {"slot": new_slot}, "output": slot_check}],
            "last_action_result": "failed", **log(state, "rescheduling", "Refused: new slot also unavailable."),
        }

    calendar_result = calendar_tool.invoke({"operation": "update", "appointment_id": state["appointment_id"], "slot": new_slot})
    crm_tool.invoke({"operation": "log_appointment", "appointment_id": state["appointment_id"], "action": "rescheduled", "detail": new_slot})
    reply = f"Theek hai, visit ab {new_slot} par move kar di gayi hai."
    return {
        "conversation_history": [{"role": "agent", "content": reply}],
        "tool_outputs": [{"tool": "calendar", "input": {"operation": "update"}, "output": calendar_result}],
        "selected_slot": {"slot": new_slot}, "appointment_status": "rescheduled", "last_action_result": "success",
        **log(state, "rescheduling", f"Rescheduled {state['appointment_id']} to {new_slot}."),
    }


def cancellation_node(state: VoiceAgentState) -> dict:
    if not state["appointment_id"]:
        reply = "Cancel karne ke liye koi active appointment nahi mili."
        return {"conversation_history": [{"role": "agent", "content": reply}], "last_action_result": "failed",
                **log(state, "cancellation", "Refused: no existing appointment to cancel.")}

    calendar_result = calendar_tool.invoke({"operation": "cancel", "appointment_id": state["appointment_id"]})
    crm_tool.invoke({"operation": "log_appointment", "appointment_id": state["appointment_id"], "action": "cancelled", "detail": "Caller requested cancellation"})
    reply = "Aap ki appointment cancel kar di gayi hai. Koi aur madad chahiye?"
    return {
        "conversation_history": [{"role": "agent", "content": reply}],
        "tool_outputs": [{"tool": "calendar", "input": {"operation": "cancel"}, "output": calendar_result}],
        "appointment_status": "cancelled", "last_action_result": "success",
        **log(state, "cancellation", f"Cancelled {state['appointment_id']}."),
    }


def email_node(state: VoiceAgentState) -> dict:
    result = email_tool.invoke({
        "appointment_id": state["appointment_id"], "client_name": state["user_profile"]["client_name"],
        "action": state["appointment_status"],
    })
    return {
        "tool_outputs": [{"tool": "email", "input": {"appointment_id": state["appointment_id"]}, "output": result}],
        **log(state, "email", f"Notified employee of {state['appointment_status']} appointment."),
    }


def goodbye_node(state: VoiceAgentState) -> dict:
    reply = "Shukriya, RealEstate Hub se rabta karne ke liye. Allah Hafiz!"
    return {"conversation_history": [{"role": "agent", "content": reply}], **log(state, "goodbye", "Call ended.")}


def route_after_intent(state: VoiceAgentState) -> str:
    return {
        "property_question": "rag", "recommendation_request": "recommendation", "book": "booking",
        "reschedule": "rescheduling", "cancel": "cancellation", "goodbye": "goodbye",
        "unclear": "intent_detection",
    }[state["intent"]]


def route_after_action(state: VoiceAgentState) -> str:
    return "email" if state.get("last_action_result") == "success" else "intent_detection"


graph_builder = StateGraph(VoiceAgentState)
for name, fn in [
    ("greeting", greeting_node), ("intent_detection", intent_detection_node), ("rag", rag_node),
    ("recommendation", recommendation_node), ("booking", booking_node), ("rescheduling", rescheduling_node),
    ("cancellation", cancellation_node), ("email", email_node), ("goodbye", goodbye_node),
]:
    graph_builder.add_node(name, fn)

graph_builder.set_entry_point("greeting")
graph_builder.add_edge("greeting", "intent_detection")
graph_builder.add_conditional_edges("intent_detection", route_after_intent, {
    "rag": "rag", "recommendation": "recommendation", "booking": "booking",
    "rescheduling": "rescheduling", "cancellation": "cancellation",
    "goodbye": "goodbye", "intent_detection": "intent_detection",
})
graph_builder.add_edge("rag", "intent_detection")
graph_builder.add_edge("recommendation", "intent_detection")
graph_builder.add_conditional_edges("booking", route_after_action, {"email": "email", "intent_detection": "intent_detection"})
graph_builder.add_conditional_edges("rescheduling", route_after_action, {"email": "email", "intent_detection": "intent_detection"})
graph_builder.add_conditional_edges("cancellation", route_after_action, {"email": "email", "intent_detection": "intent_detection"})
graph_builder.add_edge("email", "intent_detection")
graph_builder.add_edge("goodbye", END)

call_graph = graph_builder.compile()

print("Nodes:", list(call_graph.get_graph().nodes.keys()))
print()
print(call_graph.get_graph().draw_mermaid())

Nodes: ['__start__', 'greeting', 'intent_detection', 'rag', 'recommendation', 'booking', 'rescheduling', 'cancellation', 'email', 'goodbye', '__end__']

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	greeting(greeting)
	intent_detection(intent_detection)
	rag(rag)
	recommendation(recommendation)
	booking(booking)
	rescheduling(rescheduling)
	cancellation(cancellation)
	email(email)
	goodbye(goodbye)
	__end__([<p>__end__</p>]):::last
	__start__ --> greeting;
	booking -.-> email;
	booking -.-> intent_detection;
	cancellation -.-> email;
	cancellation -.-> intent_detection;
	email --> intent_detection;
	greeting --> intent_detection;
	intent_detection -.-> booking;
	intent_detection -.-> cancellation;
	intent_detection -.-> goodbye;
	intent_detection -.-> rag;
	intent_detection -.-> recommendation;
	intent_detection -.-> rescheduling;
	rag --> intent_detection;
	recommendation --> intent_detection;
	rescheduling -.-> email;
	rescheduling -

# Task 3 - Tool Integration

Six tools, each wrapped with LangChain's `@tool` decorator so they carry a schema and could
be bound to an LLM's native tool-calling in a production build â€” the nodes here call them
directly (`tool.invoke(...)`) to keep the graph deterministic and testable without a live LLM
key, the same design choice Days 1-4 made for the other integrations:

- `search_property_tool` / `availability_checker_tool` â€” property search and live
  availability, split the same way Day 2 justified (search can be fuzzy; availability must
  be authoritative).
- `rag_search_tool` â€” returns `evidence_id=None` when nothing is grounded, which is what lets
  the `rag` node abstain instead of inventing an answer.
- `calendar_tool` / `email_tool` / `crm_tool` â€” the same create/update/cancel and
  notify/log contracts from Day 4's `MockGoogleCalendarClient`, `MockEmailClient`, and CRM
  tables, reshaped as callable tools instead of client classes.

One property (`PROP-004`, DHA Skyline Heights) is deliberately marked `"sold"` and one slot
(`PROP-001` at 3pm) is deliberately pre-booked, purely so Task 4 has real guardrail failures
to catch instead of only testing the happy path.

In [3]:
from langchain_core.tools import tool
import uuid

# Reuses the property/agent shape from Day 2, with one property deliberately
# marked unavailable so Task 4's validation test has something real to catch.
PROPERTIES = [
    {"property_id": "PROP-001", "name": "Park View Residences", "area": "Bahria Town", "bedrooms": 3, "price": 38_500_000, "status": "available"},
    {"property_id": "PROP-003", "name": "DHA Garden Villas", "area": "DHA Phase 6", "bedrooms": 5, "price": 92_500_000, "status": "available"},
    {"property_id": "PROP-004", "name": "DHA Skyline Heights", "area": "DHA Phase 6", "bedrooms": 3, "price": 41_000_000, "status": "sold"},
]
EMPLOYEES = {"PROP-001": {"id": "AGENT-001", "name": "Ayesha Khan", "email": "ayesha.khan@realestatehub.example"},
             "PROP-003": {"id": "AGENT-003", "name": "Sana Ahmed", "email": "sana.ahmed@realestatehub.example"},
             "PROP-004": {"id": "AGENT-002", "name": "Hamza Malik", "email": "hamza.malik@realestatehub.example"}}
FAQS = [
    {"id": "FAQ-002", "keywords": ["price", "change"], "answer": "Asking price current listing se verify hoti hai, brochure se nahi quote ki jaati."},
    {"id": "FAQ-003", "keywords": ["documents", "rental", "application"], "answer": "Rental application ke liye CNIC, salary slip, aur previous landlord reference chahiye hote hain."},
]
# Booked slots simulate an already-full calendar; anything not listed here is free.
BUSY_SLOTS = {"3pm": {"PROP-001"}}
_calendar_store: dict[str, dict] = {}
_email_log: list[dict] = []
_crm_log: list[dict] = []


@tool
def search_property_tool(area: str | None = None, bedrooms: int | None = None, max_budget: int | None = None) -> dict:
    """Search the property catalog by area, bedroom count, and maximum budget."""
    matches = [p for p in PROPERTIES
               if (area is None or p["area"] == area)
               and (bedrooms is None or p["bedrooms"] == bedrooms)
               and (max_budget is None or p["price"] <= max_budget)]
    return {"matches": matches}


@tool
def availability_checker_tool(property_ids: list[str] | None = None, property_id: str | None = None, slot: str | None = None) -> dict:
    """Check live availability. Pass property_ids for a listing-status batch check,
    or property_id + slot for a specific viewing-time check."""
    if slot is not None:
        busy_for_slot = BUSY_SLOTS.get(slot, set())
        is_available = property_id not in busy_for_slot
        alternatives = [s for s in ("11am", "2pm", "4pm", "6pm") if property_id not in BUSY_SLOTS.get(s, set())]
        return {"slot_available": is_available, "alternative_slots": alternatives}
    ids = property_ids or []
    return {"availability": {pid: next(p["status"] for p in PROPERTIES if p["property_id"] == pid) == "available" for pid in ids}}


@tool
def rag_search_tool(query: str) -> dict:
    """Semantic FAQ/brochure lookup. Returns evidence_id=None when nothing is grounded,
    so the caller node can abstain instead of guessing."""
    text = query.lower()
    for faq in FAQS:
        if any(kw in text for kw in faq["keywords"]):
            return {"answer": faq["answer"], "evidence_id": faq["id"]}
    return {"answer": None, "evidence_id": None}


@tool
def calendar_tool(operation: str, client_name: str = "", client_phone: str = "", property_id: str = "",
                   slot: str = "", appointment_id: str = "") -> dict:
    """Create, update, or cancel a Google-Calendar-style appointment (mock)."""
    if operation == "create":
        appointment_id = f"APPT-{uuid.uuid4().hex[:8]}"
        _calendar_store[appointment_id] = {"property_id": property_id, "client_name": client_name,
                                            "client_phone": client_phone, "slot": slot, "status": "confirmed"}
        return {"appointment_id": appointment_id, "status": "confirmed"}
    if operation == "update":
        _calendar_store[appointment_id]["slot"] = slot
        return {"appointment_id": appointment_id, "status": "confirmed", "slot": slot}
    if operation == "cancel":
        _calendar_store[appointment_id]["status"] = "cancelled"
        return {"appointment_id": appointment_id, "status": "cancelled"}
    raise ValueError(f"unknown operation {operation}")


@tool
def email_tool(appointment_id: str, client_name: str, action: str) -> dict:
    """Notify the assigned employee by email about a booked/rescheduled/cancelled visit."""
    property_id = _calendar_store.get(appointment_id, {}).get("property_id")
    employee = EMPLOYEES.get(property_id, {"name": "Unassigned", "email": "unassigned@realestatehub.example"})
    message = {"to": employee["email"], "subject": f"[{action.upper()}] {client_name} â€” {appointment_id}"}
    _email_log.append(message)
    return {"sent_to": employee["email"], "message_id": f"MSG-{uuid.uuid4().hex[:8]}"}


@tool
def crm_tool(operation: str, appointment_id: str, action: str, detail: str) -> dict:
    """Log an appointment lifecycle event to the CRM."""
    entry = {"appointment_id": appointment_id, "action": action, "detail": detail}
    _crm_log.append(entry)
    return {"logged": True, "row_count": len(_crm_log)}


print("Tools registered:", [t.name for t in
      [search_property_tool, availability_checker_tool, rag_search_tool, calendar_tool, email_tool, crm_tool]])

# Smoke-test each tool directly before trusting the graph to call them.
print(search_property_tool.invoke({"area": "DHA Phase 6", "max_budget": 100_000_000}))
print(availability_checker_tool.invoke({"property_ids": ["PROP-003", "PROP-004"]}))
print(availability_checker_tool.invoke({"property_id": "PROP-001", "slot": "3pm"}))
print(rag_search_tool.invoke({"query": "can the asking price change"}))
print(rag_search_tool.invoke({"query": "do you guarantee 30 percent investment return"}))

Tools registered: ['search_property_tool', 'availability_checker_tool', 'rag_search_tool', 'calendar_tool', 'email_tool', 'crm_tool']
{'matches': [{'property_id': 'PROP-003', 'name': 'DHA Garden Villas', 'area': 'DHA Phase 6', 'bedrooms': 5, 'price': 92500000, 'status': 'available'}, {'property_id': 'PROP-004', 'name': 'DHA Skyline Heights', 'area': 'DHA Phase 6', 'bedrooms': 3, 'price': 41000000, 'status': 'sold'}]}
{'availability': {'PROP-003': True, 'PROP-004': False}}
{'slot_available': False, 'alternative_slots': ['11am', '2pm', '4pm', '6pm']}
{'answer': 'Asking price current listing se verify hoti hai, brochure se nahi quote ki jaati.', 'evidence_id': 'FAQ-002'}
{'answer': None, 'evidence_id': None}


With the tools in place, the graph compiled in Task 2 is now fully callable. The
scripted conversation below reuses the exact multi-turn memory pattern from Day 3's design
("Budget 3 crore hai" -> "DHA mein kya options hain?" -> "Us se sasti koi option?"): budget
and area are captured on turn 1, carried silently through every later turn, a booking
attempt hits an already-taken slot, and the agent offers real alternatives instead of just
failing â€” all driven by one `call_graph.invoke()`.

In [4]:
conversation = [
    "Budget 4 crore hai, Bahria Town mein ghar chahiye",  # sets budget=40,000,000, area=Bahria Town
    "3 bedroom options dikhao",                            # recommendation_request -> matches PROP-001
    "3pm par visit book kar dain",                          # booking -> hits the busy slot, should be refused
    "4pm theek hai",                                        # booking retry -> should succeed
    "shukriya",                                             # goodbye
]

state = initial_state("Bilal Farooq", "+92-321-5551042", conversation)
final_state = call_graph.invoke(state, config={"recursion_limit": 50})

print("=== Full transcript ===")
for msg in final_state["conversation_history"]:
    print(f"[{msg['role']:>6}] {msg['content']}")

print("\n=== Final state summary ===")
print("intent:", final_state["intent"])
print("budget:", final_state["budget"])
print("property_preferences:", final_state["property_preferences"])
print("matched_property_id:", final_state["matched_property_id"])
print("appointment_id:", final_state["appointment_id"])
print("appointment_status:", final_state["appointment_status"])
print("selected_slot:", final_state["selected_slot"])

=== Full transcript ===
[ agent] Assalam-o-alaikum, main property team ka AI assistant hoon. Aap kis type ki property dekh rahe hain?
[caller] Budget 4 crore hai, Bahria Town mein ghar chahiye
[ agent] Park View Residences available hai Bahria Town mein, 3 bedroom, asking price 38,500,000 PKR. Visit book karwana chahain gay?
[caller] 3 bedroom options dikhao
[ agent] Park View Residences available hai Bahria Town mein, 3 bedroom, asking price 38,500,000 PKR. Visit book karwana chahain gay?
[caller] 3pm par visit book kar dain
[ agent] 3pm par slot already book hai. Available slots hain: 11am, 2pm, 4pm, 6pm. In mein se konsa theek rahega?
[caller] 4pm theek hai
[ agent] Confirm ho gaya â€” 4pm par aap ki visit book hai. Confirmation email employee ko chali jayegi.
[caller] shukriya
[ agent] Shukriya, RealEstate Hub se rabta karne ke liye. Allah Hafiz!

=== Final state summary ===
intent: goodbye
budget: 40000000
property_preferences: {'area': 'Bahria Town', 'bedrooms': 3}
matched_proper

# Task 4 - Validation

Three scenarios, each targeting one required guardrail:

1. **Never recommend an unavailable property** â€” the search tool matches DHA Skyline
   Heights on area/bedrooms/budget, but `availability_checker_tool` reports it `sold`, so
   `recommendation_node` filters it out before it ever reaches the caller.
2. **Never book an unavailable slot** â€” a second caller asks for Park View Residences at
   3pm, the slot Task 3's scenario already filled; `booking_node` refuses and offers real
   alternatives instead of creating a conflicting calendar event.
3. **Ask for clarification instead of guessing** â€” a caller asks about a guaranteed 30%
   investment return. No FAQ backs that claim, so `rag_search_tool` returns
   `evidence_id=None` and the `rag` node offers a specialist callback rather than fabricating
   reassurance â€” the same abstention behavior Day 2's hallucination eval required.

Each assertion checks the actual state LangGraph produced, not just the printed message, so
a regression in the routing logic would fail loudly here rather than just reading oddly in
the transcript.

In [5]:
print("--- Validation 1: never recommend an unavailable property ---")
unavailable_pref_state = initial_state("Noor Fatima", "+92-333-5552091", [
    "DHA mein 3 bedroom ghar chahiye, budget 5 crore hai",
])
result_1 = call_graph.invoke(unavailable_pref_state, config={"recursion_limit": 50})
recommended_ids = {t["output"].get("property_id") for t in result_1["tool_outputs"] if t["tool"] == "calendar"}
searched = next(t["output"]["matches"] for t in result_1["tool_outputs"] if t["tool"] == "search_property")
sold_ids_in_search = {p["property_id"] for p in searched if p["status"] == "sold"}
assert result_1["matched_property_id"] != "PROP-004", "must never recommend the sold DHA Skyline Heights listing"
print(f"Search matched {[p['property_id'] for p in searched]} (includes sold {sold_ids_in_search}), "
      f"but the agent recommended: {result_1['matched_property_id']} â€” sold listing correctly filtered out.")
print(f"Agent said: \"{result_1['conversation_history'][-2]['content']}\"")

print("\n--- Validation 2: never book an unavailable slot ---")
double_book_state = initial_state("Second Caller", "+92-300-9999999", [
    "Budget 4 crore hai, Bahria Town mein ghar chahiye",
    "3 bedroom options dikhao",
    "3pm par visit book kar dain",
])
result_2 = call_graph.invoke(double_book_state, config={"recursion_limit": 50})
assert result_2["appointment_status"] == "none", "a conflicting slot must never produce a booking"
assert result_2["appointment_id"] is None
print(f"appointment_status={result_2['appointment_status']!r}, appointment_id={result_2['appointment_id']!r} "
      "â€” booking correctly refused instead of double-booking Ayesha's 3pm slot.")
print(f"Agent said: \"{result_2['conversation_history'][-2]['content']}\"")

print("\n--- Validation 3: ask for clarification instead of guessing ---")
unsupported_state = initial_state("Talha Nadeem", "+92-345-5552003", [
    "Kya aap 30 percent guaranteed investment return dete hain?",
])
result_3 = call_graph.invoke(unsupported_state, config={"recursion_limit": 50})
rag_call = next(t for t in result_3["tool_outputs"] if t["tool"] == "rag_search")
assert rag_call["output"]["evidence_id"] is None, "no FAQ backs a guaranteed-return claim"
assert "verified information" in result_3["conversation_history"][-2]["content"] or \
       "specialist" in result_3["conversation_history"][-2]["content"]
print(f"No grounded evidence for the claim (evidence_id={rag_call['output']['evidence_id']}) â€” "
      "agent asked to escalate instead of answering.")
print(f"Agent said: \"{result_3['conversation_history'][-2]['content']}\"")

--- Validation 1: never recommend an unavailable property ---
Search matched ['PROP-004'] (includes sold {'PROP-004'}), but the agent recommended: None â€” sold listing correctly filtered out.
Agent said: "Is budget aur area mein abhi koi available option nahi hai. Kya aap budget thora barha saktay hain, ya nearby area consider karain gay?"

--- Validation 2: never book an unavailable slot ---
appointment_status='none', appointment_id=None â€” booking correctly refused instead of double-booking Ayesha's 3pm slot.
Agent said: "3pm par slot already book hai. Available slots hain: 11am, 2pm, 4pm, 6pm. In mein se konsa theek rahega?"

--- Validation 3: ask for clarification instead of guessing ---
No grounded evidence for the claim (evidence_id=None) â€” agent asked to escalate instead of answering.
Agent said: "Sorry, is baare mein mujhe verified information nahi mil rahi. Main specialist se confirm karwa kar callback arrange kar sakta hoon â€” theek hai?"


# Task 5 - State Logging

Every node appends one `TraceEntry` to `execution_trace` via the shared `log()` helper from
Task 2 â€” node name, the intent in effect when it ran, the appointment status at that point,
and a human-readable note of what the node actually decided. `print_annotated_trace` turns
that into a readable, numbered call log for each of the four conversations run above.

The node-visit counter at the end is the same idea a real observability dashboard would use
this trace for in production: not just replaying one call, but rolling many calls' traces up
into a metric â€” e.g. "how often does `intent_detection` self-loop on `unclear`," which is a
direct proxy for how often the NLU layer needs a clarifying question.

In [6]:
def print_annotated_trace(state: VoiceAgentState, label: str) -> None:
    print(f"=== Execution trace: {label} ===")
    for i, entry in enumerate(state["execution_trace"], start=1):
        print(f"{i:>2}. [{entry['node']:<15}] intent={entry['entered_intent'] or '-':<22} "
              f"status={entry['appointment_status']:<11} | {entry['note']}")
    print(f"Total node visits: {len(state['execution_trace'])}\n")


print_annotated_trace(final_state, "successful booking call (Task 3 scenario)")
print_annotated_trace(result_1, "Validation 1 - unavailable-property call")
print_annotated_trace(result_2, "Validation 2 - conflicting-slot call")
print_annotated_trace(result_3, "Validation 3 - unsupported-claim call")

# A trace is only useful for debugging if it's queryable, not just printable â€”
# summarize node-visit frequency across all four calls, the way a real
# observability dashboard would roll traces up into a metric.
from collections import Counter
all_traces = final_state["execution_trace"] + result_1["execution_trace"] + result_2["execution_trace"] + result_3["execution_trace"]
node_visit_counts = Counter(entry["node"] for entry in all_traces)
print("Node visit counts across all 4 calls:", dict(node_visit_counts))

=== Execution trace: successful booking call (Task 3 scenario) ===
 1. [greeting       ] intent=-                      status=none        | Opened the call.
 2. [intent_detection] intent=-                      status=none        | Parsed intent=recommendation_request from: "Budget 4 crore hai, Bahria Town mein ghar chahiye"
 3. [recommendation ] intent=recommendation_request status=none        | Recommended PROP-001 (available, within budget).
 4. [intent_detection] intent=recommendation_request status=none        | Parsed intent=recommendation_request from: "3 bedroom options dikhao"
 5. [recommendation ] intent=recommendation_request status=none        | Recommended PROP-001 (available, within budget).
 6. [intent_detection] intent=recommendation_request status=none        | Parsed intent=book from: "3pm par visit book kar dain"
 7. [booking        ] intent=book                   status=none        | Refused: requested slot unavailable (offered alternatives instead of guessing).
 8. 

**Design notes:** the validation and unsupported-claim calls are visibly shorter (5
node visits) than the successful booking call (12) â€” the trace makes it easy to see *why*
without re-reading the full transcript: both end after a single refused/abstained action
instead of proceeding through booking and email. Day 6's evaluation suite can consume these
traces directly as its ground truth for "did the agent take the right path," rather than
parsing transcripts.